In [4]:
#Assignment 3 — IoT Sensor Data Analysis 
#Dataset iot_sensor_data_raw.csv 
#Objective 
#You are working as an IoT data analyst for a manufacturing company. 
#The company has multiple sensors installed across factories. Your task is to analyze the sensor data and identify abnormal operating conditions. 
import pandas as pd

#Part A — Data Preparation 
#1.	Load the dataset.  
dataset = pd.read_csv('../data/iot_sensor_data_raw.csv')
df = pd.DataFrame(dataset)

#2.	Display its dimensions and structure. 
df.shape
df.info()
df.describe()

#3.	Convert Timestamp into Pandas datetime. 
df['pd_DateTime'] = pd.to_datetime(df['Timestamp'])

#4.	Check whether the timestamps are correctly ordered. 
is_ordered = df['pd_DateTime'].is_monotonic_increasing
print(f"Are timestamps correctly ordered? {is_ordered}")

#5.	Identify missing sensor readings.
# Returns the total number of missing values for every column
print(df.isna().sum())

#6.	Calculate the percentage of missing values for each sensor.  
missing_percentage = df.isna().mean() * 100

# Display the results neatly sorted from highest to lowest
print("--- Missing Data Percentage By Column ---")
print(missing_percentage.sort_values(ascending=False).round(2).astype(str) + '%')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Timestamp       20000 non-null  object 
 1   Device_ID       20000 non-null  object 
 2   Temperature     19701 non-null  float64
 3   Humidity        19700 non-null  float64
 4   Pressure        19700 non-null  float64
 5   Vibration       19702 non-null  float64
 6   Battery_Level   20000 non-null  float64
 7   Location        20000 non-null  object 
 8   Machine_Status  20000 non-null  object 
dtypes: float64(5), object(4)
memory usage: 1.4+ MB
Are timestamps correctly ordered? False
Timestamp           0
Device_ID           0
Temperature       299
Humidity          300
Pressure          300
Vibration         298
Battery_Level       0
Location            0
Machine_Status      0
pd_DateTime         0
dtype: int64
--- Missing Data Percentage By Column ---
Pressure           1.5%
Humidi

In [19]:
#Part B — Data Cleaning 
#Battery 	Status 
#>= 50 	Healthy 

#7.	Handle missing sensor values using an appropriate method. 
df.fillna(0, inplace=True)

#8.	Identify abnormal temperature readings.  
# Calculate quartiles (Q1 = 25th percentile, Q3 = 75th percentile)
Q1 = df['Temperature'].quantile(0.25)
Q3 = df['Temperature'].quantile(0.75)
IQR = Q3 - Q1

# Define boundaries (1.5 * IQR is the standard multiplier)
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out the 0s from missing data so they aren't flagged as temperature anomalies
is_anomaly = ((df['Temperature'] < lower_bound) | (df['Temperature'] > upper_bound)) & (df['Temperature'] != 0)

# Extract abnormal readings
anomalies = df[is_anomaly]
print(f"Found {len(anomalies)} abnormal temperature readings.")

#9.	Identify abnormal vibration readings.  
Q1 = df['Vibration'].quantile(0.25)
Q3 = df['Vibration'].quantile(0.75)
IQR = Q3 - Q1

# Define boundaries (1.5 * IQR is the standard multiplier)
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Filter out the 0s from missing data so they aren't flagged as temperature anomalies
is_anomaly = ((df['Vibration'] < lower_bound) | (df['Vibration'] > upper_bound)) & (df['Vibration'] != 0)

# Extract abnormal readings
anomalies = df[is_anomaly]
print(f"Found {len(anomalies)} abnormal Vibration readings.")

#10.	Identify machines with critically low battery levels.
CRITICAL_THRESHOLD = 15.0
low_level_battries = (df['Battery_Level'] < CRITICAL_THRESHOLD)
critical_events = df[low_level_battries]
critical_events = critical_events.sort_values(by='Battery_Level', ascending=True)

print("--- All Critical Battery Logs ---")
print(critical_events.head())
print("rows with critical battery level ", len(critical_events)) 


#11.	Explain how you decided what constitutes an abnormal reading.  
#used statistical thresholds the Interquartile Range (IQR) to calculate abnormal readings.

Found 158 abnormal temperature readings.
Found 153 abnormal Vibration readings.
--- All Critical Battery Logs ---
                 Timestamp Device_ID  Temperature  Humidity  Pressure  \
4861   2026-05-04 14:30:00   DEV_008        64.96     57.59   1027.14   
2328   2026-06-16 06:00:00   DEV_007        64.20     75.38   1026.62   
8158   2026-05-09 12:45:00   DEV_002        73.10     81.95   1024.76   
18694  2026-06-30 14:30:00   DEV_006        73.28     17.58   1004.17   
9727   2026-03-11 02:15:00   DEV_001        76.26     60.07    995.10   

       Vibration  Battery_Level   Location Machine_Status         pd_DateTime  
4861        2.38       2.011730  Factory_C         Normal 2026-05-04 14:30:00  
2328        2.21       2.146086  Factory_A         Normal 2026-06-16 06:00:00  
8158        0.00       2.154615  Factory_B         Normal 2026-05-09 12:45:00  
18694       0.00       2.201033  Factory_B         Normal 2026-06-30 14:30:00  
9727        2.05       2.257059  Factory_C     

In [6]:
#Part C — Time-Series Analysis 
#12.	Calculate the average temperature by hour.  
hourly_averages = df.groupby(df['pd_DateTime'].dt.hour)['Temperature'].mean()
print("--- Average Temperature By Hour ---")
print(hourly_averages.round(2))

#13.	Calculate the average temperature for each device.  
avg_temp_by_device = df.groupby(df['Device_ID'])['Temperature'].mean()
print("--- Average Temperature By Device ---")
print(avg_temp_by_device.round(2))

#14.	Calculate the average vibration for each device.
avg_vib_for_device = df.groupby(df['Device_ID'])['Vibration'].mean()
print("--- Average Vibration By Device ---")
print(avg_vib_for_device.round(2))

#15.	Find the maximum temperature recorded by each device.  
max_temp_by_device = df.groupby(df['Device_ID'])['Temperature'].max()
print("--- Max Temerature By Device ---")
print(max_temp_by_device.round(2))

#16.	Find the minimum battery level for each device.  
min_battery_level_by_device = df.groupby(df['Device_ID'])['Battery_Level'].min()
print("--- Min Battery level By Device ---")
print(min_battery_level_by_device.round(2))

#17.	Determine which device has the highest average vibration.
#max_index = avg_vib_for_device.
print("device with the highest average vibration :", avg_vib_for_device.idxmax())

#18.	Determine which factory has the highest average temperature.  
factory_with_highest_avg_temp = df.groupby(df['Location'])['Temperature'].mean().idxmax()
highest_avg_temp = df.groupby(df['Location'])['Temperature'].mean().max()
print('factory with highest avg temperature ::', factory_with_highest_avg_temp)
print('Temperature ::', highest_avg_temp)


--- Average Temperature By Hour ---
pd_DateTime
0     70.08
1     70.55
2     69.92
3     69.71
4     69.81
5     70.27
6     70.25
7     70.66
8     70.39
9     70.98
10    69.93
11    69.89
12    69.99
13    70.13
14    69.83
15    70.22
16    69.97
17    70.56
18    70.22
19    70.00
20    70.14
21    70.14
22    71.19
23    70.41
Name: Temperature, dtype: float64
--- Average Temperature By Device ---
Device_ID
DEV_001    70.25
DEV_002    70.38
DEV_003    69.93
DEV_004    70.58
DEV_005    70.12
DEV_006    70.01
DEV_007    70.15
DEV_008    70.32
Name: Temperature, dtype: float64
--- Average Vibration By Device ---
Device_ID
DEV_001    2.55
DEV_002    2.56
DEV_003    2.54
DEV_004    2.52
DEV_005    2.49
DEV_006    2.55
DEV_007    2.52
DEV_008    2.53
Name: Vibration, dtype: float64
--- Max Temerature By Device ---
Device_ID
DEV_001    159.75
DEV_002    151.82
DEV_003    157.60
DEV_004    151.97
DEV_005    152.97
DEV_006    136.00
DEV_007    153.88
DEV_008    153.62
Name: Temperature, 

In [7]:
#Part D — Create New Variables 
#19.	Create a Battery_Status column:  
#20–49         	Moderate 
#<20 	Critical 
import numpy as np
bins = [-np.inf, 20, 49, np.inf]
labels = ['Critical', 'Moderate', 'Good']

# Create the new column
df['Battery_Status'] = pd.cut(df['Battery_Level'], bins=bins, labels=labels)
  
#20.	Create a Temperature_Status column. 
mean_temprature = df['Temperature'].mean()
print('mean_temprature :', mean_temprature)
bins = [-np.inf, 0, mean_temprature, np.inf]
labels = ['Low', 'Normal', 'High']
df['Temperature_Status'] = pd.cut(df['Temperature'], bins=bins, labels=labels)

#21.	Create a Vibration_Status column.  
avg_vib = df['Vibration'].mean()
print('avg_vib :', avg_vib)
labels = ['Low', 'Moderate', 'High']
df['Vibration_Status'] = pd.cut(df['Temperature'], bins=3, labels=labels)

#22.	Create an overall Machine_Health column:  

#Normal 
#Warning 
#Critical 
#based on appropriate sensor conditions. 

conditions = [
    # Rule 1: Any critical sensor reading forces a Critical machine status
    (df['Temperature_Status'] == 'High') | (df['Battery_Status'] == 'Critical'),
    
    # Rule 2: Any warning signs escalate the machine to a Warning status
    (df['Temperature_Status'] == 'Low') | (df['Vibration_Status'] == 'Low'),
    
    # Rule 3: Both sensors are perfectly healthy
    (df['Temperature_Status'] == 'Normal') & (df['Vibration_Status'] != 'Low') & (df['Battery_Status'] != 'Critical')
]

# 2. Assign corresponding health output labels matching the conditions above
health_labels = ['Critical', 'Warning', 'Normal']

# 3. Create the composite column (default to 'Unknown' if rows don't match any rule)
df['Machine_Health'] = np.select(conditions, health_labels, default='Unknown')
df.head()

mean_temprature : 70.21840629770045
avg_vib : 2.534875494781429


,Timestamp,Device_ID,Temperature,Humidity,Pressure,Vibration,Battery_Level,Location,Machine_Status,pd_DateTime,Battery_Status,Temperature_Status,Vibration_Status,Machine_Health
0,2026-04-21 22:30:00,DEV_005,65.89,62.70,1007.96,2.22,46.51,Factory_A,Warning,2026-04-21 22:30:00,Moderate,Normal,Low,Warning
1,2026-01-22 06:15:00,DEV_007,70.65,70.95,1021.85,2.73,57.04,Factory_A,Normal,2026-01-22 06:15:00,Good,High,Low,Critical
2,2026-04-01 07:00:00,DEV_005,81.01,60.49,1009.82,2.15,11.65,Factory_A,Normal,2026-04-01 07:00:00,Critical,High,Moderate,Critical
3,2026-01-12 14:30:00,DEV_007,74.17,68.76,1001.42,2.03,17.36,Factory_C,Critical,2026-01-12 14:30:00,Critical,High,Low,Critical
4,2026-05-25 19:30:00,DEV_008,74.04,54.32,1014.34,NaN,27.04,Factory_C,Normal,2026-05-25 19:30:00,Moderate,High,Low,Critical


In [15]:
#Part E — Advanced Analysis 
#23.	Find devices that have experienced critical conditions more than five times. 
critical_data = df[df['Machine_Health'] =='Critical']
critical_device_grouped = critical_data.groupby('Device_ID').size()
devices_with_critical_conditions = critical_device_grouped[critical_device_grouped > 5]
print('devices_with_critical_conditions ::', devices_with_critical_conditions)

#24.	Find the factory with the highest number of abnormal sensor readings. 
# Filter for readings where Machine_Status is not Normal (Warning or Critical)
abnormal_data = df[df['Machine_Status'] != 'Normal']

# Group by 'Location' (Factory) and count the occurrences
factory_abnormal_counts = abnormal_data.groupby('Location').size()

# Find the factory with the highest count
highest_abnormal_factory = factory_abnormal_counts.idxmax()
highest_abnormal_count = factory_abnormal_counts.max()

print(f"Factory with highest abnormal readings: {highest_abnormal_factory} ({highest_abnormal_count} readings)")


#25.	Calculate the percentage of time each device operates under warning/critical conditions.  
# Count total readings per device
total_readings = df.groupby('Device_ID').size()

# Filter for Warning/Critical and count those readings per device
unhealthy_readings = df[df['Machine_Health'].isin(['Warning', 'Critical'])].groupby('Device_ID').size()

# Align the data (fill missing devices with 0 if they had no unhealthy readings) and calculate %
device_abnormal_percentage = (unhealthy_readings.reindex(total_readings.index, fill_value=0) / total_readings) * 100

print("Percentage of time under warning/critical conditions per device:\n")
print(device_abnormal_percentage.round(2).astype(str) + '%')


#26.	Identify the device that requires the highest maintenance priority. 

# Step 1: Calculate the percentage of time spent in Warning/Critical health (from Q25)
total_readings = df.groupby('Device_ID').size()
unhealthy_readings = df[df['Machine_Health'].isin(['Warning', 'Critical'])].groupby('Device_ID').size()
unhealthy_pct = (unhealthy_readings.reindex(total_readings.index, fill_value=0) / total_readings) * 100

# Step 2: Calculate the percentage of time the battery is in a 'Critical' state
critical_battery_readings = df[df['Battery_Status'] == 'Critical'].groupby('Device_ID').size()
battery_crit_pct = (critical_battery_readings.reindex(total_readings.index, fill_value=0) / total_readings) * 100

# Step 3: Combine them into a DataFrame to determine priority
maintenance_df = pd.DataFrame({
    'Unhealthy_Health_Pct': unhealthy_pct,
    'Critical_Battery_Pct': battery_crit_pct
})

# Create a Priority Score (Weighting health risk heavily, backed by battery risk)
maintenance_df['Priority_Score'] = (maintenance_df['Unhealthy_Health_Pct'] * 0.7) + (maintenance_df['Critical_Battery_Pct'] * 0.3)

# Step 4: Identify the top priority device
highest_priority_device = maintenance_df['Priority_Score'].idxmax()

print(f"Device requiring highest maintenance priority: {highest_priority_device}")
print("\nTop maintenance priority leaderboard:")
print(maintenance_df.sort_values(by='Priority_Score', ascending=False).round(2))


devices_with_critical_conditions :: Device_ID
DEV_001    1357
DEV_002    1399
DEV_003    1302
DEV_004    1406
DEV_005    1378
DEV_006    1293
DEV_007    1312
DEV_008    1391
dtype: int64
Factory with highest abnormal readings: Factory_A (1423 readings)
Percentage of time under warning/critical conditions per device:

Device_ID
DEV_001    98.45%
DEV_002    98.81%
DEV_003     98.6%
DEV_004    98.96%
DEV_005     98.8%
DEV_006     98.9%
DEV_007    98.72%
DEV_008    98.51%
dtype: object
Device requiring highest maintenance priority: DEV_004

Top maintenance priority leaderboard:
           Unhealthy_Health_Pct  Critical_Battery_Pct  Priority_Score
Device_ID                                                            
DEV_004                   98.96                 12.43           73.00
DEV_006                   98.90                 12.09           72.86
DEV_005                   98.80                 12.05           72.77
DEV_002                   98.81                 11.90           72.74